# 03.4 Line Continuation

Python normally treats the end of a line as the end of a statement. When an
expression is too long to fit comfortably, you need a way to spread it over
several lines.

There are two mechanisms. One is used constantly; the other should be avoided.

## Theory

### Why line length matters at all

PEP 8 caps lines at **79 characters**; many modern projects use **88** (the
`black` default) or 100. The reason is not nostalgia for narrow terminals — it
is that side-by-side diffs, code review panes, and split editors all become
unusable when lines run long.

### Mechanism 1: implicit continuation (preferred)

Inside any bracket — `()`, `[]`, `{}` — Python **ignores line breaks entirely**.
The statement continues until the bracket closes.

```python
total = (first_value
         + second_value
         + third_value)
```

This works because the tokenizer tracks bracket depth. While depth is greater
than zero, newlines are not statement terminators. You met the same rule in 03.2:
indentation is also ignored inside brackets.

### Mechanism 2: the backslash (avoid)

A backslash at the very end of a line joins it to the next.

```python
total = first_value + \
        second_value
```

It works, but it is fragile: **a single trailing space after the backslash breaks
it**, and that space is invisible. PEP 8 says prefer brackets.

### The one place a backslash is unavoidable

`with` statements managing multiple resources could not be wrapped in brackets
before Python 3.10. Since 3.10, parenthesised context managers are supported, so
even this case is now solved.

In modern Python, you effectively never need a backslash.

In [ ]:
# IMPLICIT CONTINUATION - the tokenizer ignores newlines inside brackets.

first_value = 100
second_value = 250
third_value = 75

# Parentheses let the expression span lines. The operators lead each line,
# which PEP 8 recommends because it makes the structure scannable.
total = (first_value
         + second_value
         + third_value)

print("Sum across three lines:", total)

# The same applies to a list literal.
weekdays = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
]
print("List across six lines:", len(weekdays), "days")

# And to a dictionary.
settings = {
    "host": "localhost",
    "port": 8080,
    "debug": True,
}
print("Dict across five lines:", settings)

# And to a function call with many arguments.
message = "-".join([
    "alpha",
    "beta",
    "gamma",
])
print("Call across five lines:", message)

## The trailing comma

Notice the comma after the last item in the collections above. Python allows it,
and it is worth adopting as a habit.

**Why:** adding a new item changes only one line instead of two, so version
control diffs stay clean and merge conflicts become less likely.

In [ ]:
# WITHOUT a trailing comma, adding "Saturday" touches two lines:
#     "Friday"          ->  "Friday",
#                           "Saturday"

# WITH a trailing comma, it touches one:
#     "Friday",         ->  "Friday",
#                           "Saturday",

without_trailing = [
    "Monday",
    "Tuesday"
]

with_trailing = [
    "Monday",
    "Tuesday",
]

# Both produce an identical list - the comma is purely cosmetic to Python.
print("without trailing comma:", without_trailing)
print("with trailing comma:   ", with_trailing)
print("identical?", without_trailing == with_trailing)

# The trailing comma is allowed in lists, dicts, sets, tuples and calls.
print("")
print("Allowed in every collection type:")
print("   list: ", [1, 2, 3,])
print("   tuple:", (1, 2, 3,))
print("   set:  ", {1, 2, 3,})
print("   dict: ", {"a": 1, "b": 2,})

### The one place a trailing comma changes meaning

A single value in parentheses is **not** a tuple. Adding a trailing comma is what
makes it one. This is the single-element tuple trap from Chapter 12, and it is
worth meeting early.

In [ ]:
# Parentheses alone do NOT create a tuple - they just group.
not_a_tuple = (5)
print("(5)    is", type(not_a_tuple).__name__, "with value", not_a_tuple)

# The COMMA is what creates a tuple.
actually_a_tuple = (5,)
print("(5,)   is", type(actually_a_tuple).__name__, "with value", actually_a_tuple)

# You do not even need the brackets - the comma alone is enough.
also_a_tuple = 5,
print("5,     is", type(also_a_tuple).__name__, "with value", also_a_tuple)

print("")
print("This catches people out when a function expects a tuple:")

def show_length(value):
    """Report how many items a value contains."""
    return len(value)

print("   len((5,)) =", show_length((5,)))

try:
    show_length((5))
except TypeError as error:
    print("   len((5))  ->", error)

## Backslash continuation, and why to avoid it

The backslash works. The problem is that its correctness depends on an
**invisible** character.

In [ ]:
NEWLINE = chr(10)
BACKSLASH = chr(92)

# A correct backslash continuation: nothing after the backslash.
good_source = ("total = 1 + " + BACKSLASH + NEWLINE
               + "        2" + NEWLINE)

# The same thing with ONE trailing space after the backslash.
bad_source = ("total = 1 + " + BACKSLASH + " " + NEWLINE
              + "        2" + NEWLINE)

for label, source in [("no trailing space", good_source),
                      ("one trailing space", bad_source)]:
    # repr() exposes the invisible character so we can see the difference.
    print(label + ":")
    print("   source:", repr(source))
    try:
        compile(source, "<demo>", "exec")
        print("   compiles fine")
    except SyntaxError as error:
        print("   SyntaxError:", error.msg)
    print("")

print("The two sources differ by one invisible character.")
print("This is why PEP 8 prefers brackets.")

## Where brackets already exist for free

Often you do not need to add brackets — the syntax already provides them.

In [ ]:
values = [10, 20, 30, 40, 50]

# A function call already has brackets, so arguments can span lines.
result = max(
    values[0],
    values[-1],
    sum(values) // len(values),
)
print("call spanning lines:", result)

# A comprehension already has brackets.
labelled = [
    f"item-{value}"
    for value in values
    if value > 20
]
print("comprehension spanning lines:", labelled)

# A conditional expression can be wrapped in parentheses.
status = (
    "high"
    if sum(values) > 100
    else "low"
)
print("conditional spanning lines:", status)

# Chained method calls wrap naturally inside parentheses.
cleaned = (
    "  Hello World  "
    .strip()
    .lower()
    .replace(" ", "_")
)
print("chained methods spanning lines:", repr(cleaned))

## Implicit string concatenation

Two string literals written next to each other are joined **at compile time**.
Combined with bracket continuation, this is how long messages are formatted.

Be careful: a missing comma in a list of strings silently triggers this and
merges two entries into one. It is a genuine source of bugs.

In [ ]:
# Adjacent string literals are concatenated by the compiler.
message = (
    "Dear customer, "
    "your order has shipped. "
    "Thank you for your business."
)
print("Joined message:")
print("   ", message)

# Proof it happens at COMPILE time, not runtime - there is no + instruction.
import dis
print("")
print("Bytecode for two adjacent literals:")
dis.dis(compile('x = "abc" "def"', "<demo>", "exec"))

print("")
print("Python stored one constant, 'abcdef'. No concatenation at runtime.")

# THE TRAP: a missing comma in a list silently merges two items.
print("")
with_comma = ["alpha", "beta", "gamma"]
missing_comma = ["alpha", "beta" "gamma"]

print("with every comma: ", with_comma, "->", len(with_comma), "items")
print("one comma missing:", missing_comma, "->", len(missing_comma), "items")
print("")
print("No error was raised. 'beta' and 'gamma' were silently joined.")
print("Linters catch this - another reason to run one.")

## Style: where to break a long line

Given a choice of break points, some read far better than others. PEP 8's rule is
to break **before** a binary operator, so the operator starts the new line.

In [ ]:
price = 100
quantity = 3
discount = 0.1
tax_rate = 0.18
shipping = 50

# POOR: operators trail at the end of lines, hard to scan.
total_poor = (price * quantity -
              price * quantity * discount +
              shipping)

# GOOD: operators lead each line, so the structure is obvious.
total_good = (
    price * quantity
    - price * quantity * discount
    + shipping
)

print("Both give the same answer:", total_poor == total_good, "->", total_good)

print("")
print("PEP 8 rule: break BEFORE the operator.")
print("Reason: the eye scans the left edge, so operators line up and read")
print("like the mathematics they represent.")

## Multiple context managers

Before Python 3.10, wrapping a `with` statement in parentheses was a syntax
error, which was the last real use case for the backslash. Modern Python fixes
this.

In [ ]:
import sys

print("Running Python", ".".join(str(part) for part in sys.version_info[:2]))

# Since 3.10, context managers can be parenthesised across lines:
#
#     with (
#         open("a.txt") as first,
#         open("b.txt") as second,
#     ):
#         ...
#
# Before 3.10 you needed a backslash:
#
#     with open("a.txt") as first, \
#          open("b.txt") as second:
#         ...

supports_parenthesised = sys.version_info >= (3, 10)
print("Parenthesised context managers supported?", supports_parenthesised)

print("")
print("With 3.10+ there is no remaining case where a backslash is required.")
print("Context managers are covered fully in Chapter 29.")

## Takeaways

1. Inside brackets `()`, `[]`, `{}`, Python **ignores line breaks** — this is
   implicit continuation, and it is the preferred mechanism.
2. A trailing backslash also works, but breaks on an **invisible trailing
   space**. Avoid it.
3. Add a **trailing comma** to multi-line collections — it keeps diffs to one
   line per change.
4. A trailing comma changes meaning in exactly one place: `(5)` is an int,
   `(5,)` is a tuple.
5. Adjacent string literals are joined **at compile time** — and a missing comma
   in a list triggers this silently.
6. Break **before** binary operators so they lead each line.
7. Since Python 3.10, parenthesised context managers remove the last need for a
   backslash.

## Try it yourself

1. Write a sum of five numbers across five lines using parentheses. Then try it
   with backslashes and add a trailing space to one — read the error.
2. Create a list of three strings and delete one comma. Print the length. Why is
   there no error?
3. Check `type((5))` and `type((5,))`. Then try `type(())` — what is an empty
   tuple?
4. Take a long line from your own code and rewrite it breaking before the
   operators. Which version would you rather review?